# Latent Variable Models + Variational Inference

I talk a good amount about latent variable models in the gaussian mixture model notes (data science notebook). To summarize, latent variable models involve an unknown, "latent," variable that describes some unobserved data generating process. So, unlike simpler models, where the uncertainty is captured by the noise terms, latent variable models go one step further by introducing uncertainty structurally into a latent random variable $z$. 

These latent variables might correspond to specific quantities involved in the data generation process, such as the unknown orientation of an object in three-dimensional space in the case of images, or they may be introduced simply as modelling constructs to allow much richer models to be created. If we define a joint distribution over observed and latent variables, the corresponding distribution of the observed variables alone is obtained by marginalization. This allows relatively complex marginal distributions over observed variables to be expressed in terms of more tractable joint distributions over the expanded space of observed and latent variables.

## A quick review of Bayesian inference

Before going into latent variable models, it is helpful do a quick review of Bayesian inference to clarify the "how" and, more importantly, the "why." A key task in machine learning is to take a model and the data to uncover the values of the model’s hidden params $\theta$ given the observed variables $x$. Using maximum likelihood, we obtain point estimates $\theta^* = \arg\max_\theta p(X|\theta)$. Alternatively, we can also use maximum a priori (MAP) estimation. 

Both MLE and MAP produce a single best value of $\theta$, which we can then plug into a predictive distribution $p(x|\hat{\theta}^*)$ for prediction tasks. However, **focusing solely on the statistic of the posterior distribution leads to loss of information**. While simple and computationally convenient, they don't necessarily tell us **how confident we are in that estimate**. Two very different parameter settings might produce nearly identical likelihoods, yet a point estimate would treat them as if one is definitively correct and the other does not exist.

Recall the central idea of Bayesian stats is that we need to combine prior information with new data. These decision-making systems typically have different objective functions than the likelihood. Rather than committing to a single $\hat{\theta}^*$, we maintain a full distribution over parameters, called the **posterior**. Having the full posterior distribution around can be extremely useful and leads to more robust decisions. To that end, **Bayesian inference** gives us a principled way to incorporate prior knowledge, account for side information, and incorporate structural knowledge, all of which is not easily done in the context of parameter estimation.

For a dataset $\Chi$, a parameter prior $p(\theta)$, and the likelihood function, the posterior is
$$p(\theta | \Chi) = \frac{p(\Chi | \theta)p(\theta)}{p(\Chi)} \quad\text{and}\quad p(\Chi) = \int p(\Chi|\theta) p(\theta) d \theta$$

Having the posterior distribution in this case is useful because it allows us propogate uncertainty from the parameters to the data. More specifically, given that we have a prior $p(\theta)$, the predictions become
$$p(x) = \int p(x|\theta)p(\theta)d\theta = \mathbb{E}_\theta[p(x|\theta)]$$
Notice that $\theta$ has been marginalized out -- the predictions no longer depend on any single parameter choice. This averaging effect tends to produce more robust predictions, especially when data is limited and many parameter settings are plausible.

## Latent variable models

With the Bayesian framework in mind, we can introduce latent variable models, which allow us to extend the above framework by positing a data generating process (via the latent variables) in addition to the observed data $x$. Specifically, given data $x$, the model parameters $\theta$, and the latent variables $z$, we have the joint distribution over both the observed data $x$ and the latent variable $z$:
$$p(x,z |\theta) = p(x|z,\theta)p(z)$$
This factorization encodes a natural generative story where we first sample a latent cause from a prior $p(z)$, and then generate observed data $x$ from the conditional $p(x | z,\theta)$, parameterized by $\theta$. 

It helps to be explicit about what we want here as the introduction of $z$ can get a bit confusing. Since the goal of our machine learning task is still to estimate the parameters $\theta$, we want to find the **marginal likelihood** $p(x|\theta)$. Note here that the likelihood does not depend on the latent variables. As such, we have to marginalize out the latent variables which gives us a generative factorization:

$$p(x|\theta) = \int p(x|z,\theta) p(z)dz$$

This might feel a bit counterintuitive: why would we introduce the latent variable just to get rid of it? Remember that the latent variable $z$ acts as a explanatory bookkeeping device that allows us to make the learning objective more expressive. Think of $z$ as the hidden state of the world that caused your observation. To evaluate how likely your observation is under the model, you have to consider all possible hidden states that could have produced it. Now that we have an expression for the marginal likelihood, we can perform MLE in the standard way, where

$$\hat{\theta} = \arg\max_\theta p(\Chi|\theta) = \arg\max_\theta \sum^N_{i=1}\log p(x_i|\theta)$$

We can also perform Bayesian inference via MAP by placing a prior $p(\theta)$ on the model params and using Bayes theorem to obtain the posterior distribution
$$p(\theta | \Chi) = \frac{p(\Chi | \theta)p(\theta)}{p(\Chi)}$$

To recap, we are performing a two-step procedure: (1) compute the liklihood $p(x|\theta)$ as above, and (2) use the likelihood for MLE or Bayesian inference. Of course, the marginalization step is not always anlytically tractable, which means we will need approximations. This is the motivation for the EM algorithm and variational inference, which we will see in a second.

### Latent posteriors
Additionally, latent variables also give us insight into latent explanations $p(z|\theta)$. Here, we know that the likelihood $p(\Chi | \theta)$ requires the marginalization of latent variables. So, unless we have a conjugate prior $p(z)$ for $p(x|z, \theta)$, the integral is not analytically tractable. Similar to the parameter posterior, we can compute a posterior on the latent variables according to

$$p(\theta | z) = \frac{p(z | \theta)p(\theta)}{p(z)} \quad\text{and}\quad p(z) = \int p(z|\theta) p(\theta) d \theta$$

This gives rise to a couple of use cases: some that we have already seen (PCA and GMMs) and some new (LDA).

### Non-linear latent variable model
So far, we didn't specify what the latent variables look like. In general, latent variables can be both discrete and continuous depending on the application. For clustering with GMMs for example, we assume that the latent variables represent the different possible clusters that a given data point can belong to. As such, they are modeled as discrete. 

For the non-linear latent variable models, we assume that both the data and the latent variable are continuous and multivariate (foundational to variational autoencoders, which we will see later). The prior $p(z)$ is a standard multivariate normal and the conditional likelihood $p(x|z, \phi)$ is also normally distributed, where its mean is a nonlinear function $f(z,\phi)$ and the covariance $\sigma^2 I$ is spherical:
$$p(x|z,\phi) = \mathcal{N}\left(f(z, \phi), \sigma^2 I\right)$$
Here, the function $f(\cdot)$ can be any continuous, non-linear function, but most use cases of $f$ refer to a deep neural network with parameters $\phi$. Intuitively, the model $f(z, \phi)$ describes the important aspects of the data and the remaining unmodeled aspects are ascribed to the noise $\sigma^2 I$.

Similar to above, the likelihood is $p(x|\phi)$ is found by marginalizing over the latent variable $z$:
$$
\begin{align*}
p(x|\phi) &= \int p(x, z|\phi)dz\\
&= \int p(x|z,\phi) \cdot p(z) dz \\
&= \int \mathcal{N}\left(f(z, \phi), \sigma^2 I\right) \cdot \mathcal{N}\left(0, I\right)
\end{align*}
$$

This can also be viewed as an infinite weighted sum (i.e. infinite mixture) of spherical Gaussians with different means and weights $p(z)$. In other words, each value of $z$ contributes a Gaussian centered at $f(z, \phi)$, weighted by priors $p(z)=\mathcal{N}(0, I)$. This makes the model far more expressive than a simple Gaussian, as it can represent complex, multi-modal distributions over $x$.

To train the model, we will default to MLE where we maximize the log likelihood with respect to the model parameters. For simplicity, we assume taht the variance term in the likelihood expression is known:

$$\hat{\phi} = \arg\max_\phi \left[\sum^N_{i=1}\log(p(x_i|\phi))\right]$$

where $p(x_i|\phi)$ follows the form above. Again, because the integral is intractable, there is no closed form expression for the integral.

## Evidence Lower Bound (ELBO)

To make progress, we want to define the lower bound on the log-likelihood. We start by introducing an arbitrary probability distribution $q(z)$ over the latent variables:
$$
\begin{align*}
\log(p(x|\phi)) &= \log\left(\int p(x,z|\phi)dz\right) \\
&= \log\left(\int q(z)\frac{p(x,z|\phi)}{q(z)}dz\right)
\end{align*}
$$

By Jensen's inequality for the log,
$$\log\left(\int q(z)\frac{p(x,z|\phi)}{q(z)}dz\right) \geq \int q(z)\log\left(\frac{p(x,z|\phi)}{q(z)}\right)dz$$

where the RHS is called the **evidence lower bound (ELBO)**. It gets the name because in Bayesian stats, the marginal likelihood $p(x|\phi)$ is sometimes called the **evidence**. In practice, the distribution $q(z)$ might also depend on parameters $\theta$, so we write
$$
\begin{align*}
\text{ELBO}(\phi,\theta) &= \int q(z|\theta)\log\left(\frac{p(x,z|\phi)}{q(z|\theta)}\right)dz \\
&= \mathbb{E}_{q_\theta}\left[\log\left(\frac{p(x,z|\phi)}{q(z|\theta)}\right)\right] \\
&= \mathbb{E}_{q_\theta}\left[\log p(x,z|\phi) - \log q(z|\theta)\right]
\end{align*}
$$

Let's first look at some properties of ELBO. One of the major properties is the **tightness of the bound**, particularly when $q(z|\theta) = p(z|x,\phi)$:
$$
\begin{align*}
\text{ELBO}(\phi,\theta) &= \int q(z|\theta)\log\left(\frac{p(x,z|\phi)}{q(z|\theta)}\right)dz \\
&= \int q(z|\theta)\log\left(\frac{p(z|x,\phi)p(x|\phi)}{q(z|\theta)}\right)dz \\
&= \int q(z|\theta)\log p(x|\phi)dz + \int q(z|\theta)\log\left(\frac{p(z|x,\phi)}{q(z|\theta)}\right)dz \\
&= \log p(x|\phi) + \int q(z|\theta)\log\left(\frac{p(z|x,\phi)}{q(z|\theta)}\right)dz \\
&= \log p(x|\phi) - D_{KL}\left[q(z|\theta) || p(z|x,\phi)\right]
\end{align*}
$$

This shows us that the ELBO is just the original log-likelihood (first term) minus the KL divergence. Recall that KL divergence measures the "distance" between two distributions, which means that if $q(z|\theta) = p(z|x,\phi)$, then the bound becomes tight. That is, if the distribution of the latent variable is equal to the posterior of the latent variable given the data, then we can revoer the original data point. Going back to the GMM example (refer to data science notebook), the posterior is the **responsibility** of a data point to a class/cluster, which hopefully provides some intuition. 

We also see why the ELBO corresponds to a lower bound as both the log-likelihood and the KL divergence must be non-negative:

$$\log p(x|\phi) = D_{KL}\left[q(z|\theta) || p(z|x,\phi)\right] + \text{ELBO}(\phi,\theta) \implies \log p(x|\phi) \geq \text{ELBO}(\phi,\theta)$$

It follows, then, that in order to approximate the true posterior (i.e. create a tight a bound as possible), we can minimize the KL-divergence, or equivalently maximize the ELBO. Since the KL-divergence requires finding posterior of the latent variable $p(z|x, \phi)$ (which is intractable), we look to the latter option.

### Reconstruction error perspective

Another way to consider the bound is as **reconstruction error** minus the distance to the prior.
$$
\begin{align*}
\text{ELBO}(\phi, \theta) &= \int q(z|\theta) \log\left(\frac{p_(x,z|\phi)}{q(z|\theta)}\right)dz \\
&= \int q(z|\theta) \log\left(\frac{p_(x|z, \phi)p(z)}{q(z|\theta)}\right) dz\\
&= \int q(z|\theta) \log(p(x|z, \phi))dz + \int q(z|\theta) \log\left(\frac{p(z)}{q(z|\theta)}\right)dz \\
&= \int q(z|\theta) \log(p(x|z, \phi))dz + D_{\text{KL}}\left[p(z)|q(z|\theta)\right]
\end{align*}
$$
where the first term is the reconstruction error and the second term is the distance from the prior. Here, we see that the reconstruction error as it measures the average agreement $p(x|z, \phi)$ between the latent variable and the data. We then subtract how far we are from the actual latent variable with the KL divergence. This formulation shows up a good amount and is generally used in the formulation of the VAE.